> **Archived 2025 notebook (outputs removed).** Kept for provenance only; its logs are in `results/legacy_2025/`. Use `training/notebooks/01_train_kaggle.ipynb` for the controlled study.


In [ ]:
# clone YOLOv5 repository

!git clone https://github.com/ultralytics/yolov5  # clone repo

# change directory to yolov5
%cd yolov5

In [ ]:
# install dependencies as necessary

!pip install -qr requirements.txt  # install dependencies (ignore errors)

import torch



from IPython.display import Image, clear_output  # to display images

from utils.downloads import attempt_download  # to download models/datasets



# clear_output()

print('Setup complete. Using torch %s %s' % (torch.__version__, torch.cuda.get_device_properties(0) if torch.cuda.is_available() else 'CPU'))

## Step 6: Download a Dataset



Add your Roboflow API key below to download the default money counting dataset. Alternatively, use the code provided by the Roboflow dashboard in the above step to load a custom dataset.

In [ ]:
!pip install -q roboflow

In [ ]:
import os
!pip install roboflow



from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])

project = rf.workspace("dronecv").project("mineguard-thermal")

version = project.version(15)

dataset = version.download("yolov5")

In [ ]:
%pwd

In [ ]:
dataset.location

In [ ]:
%cd /content/yolov5

In [ ]:
dataset.location

In [ ]:
# this is the YAML file Roboflow wrote for us that we're loading into this notebook with our data

%cat {dataset.location}/data.yaml

# Define Model Configuration and Architecture



We will write a yaml script that defines the parameters for our model like the number of classes, anchors, and each layer.



You do not need to edit these cells, but you may.

In [ ]:
# define number of classes based on YAML

import yaml

with open(dataset.location + "/data.yaml", 'r') as stream:

    num_classes = str(yaml.safe_load(stream)['nc'])

In [ ]:
num_classes

In [ ]:
#this is the model configuration we will use for our tutorial

%cat /kaggle/working/yolov5/models/yolov5x.yaml

In [ ]:
#customize iPython writefile so we can write variables

from IPython.core.magic import register_line_cell_magic



@register_line_cell_magic

def writetemplate(line, cell):

    with open(line, 'w') as f:

        f.write(cell.format(**globals()))

In [ ]:
%%writetemplate /kaggle/working/yolov5/models/custom_yolov5x.yaml



# parameters

nc: {num_classes}  # number of classes

depth_multiple: 1.33 # model depth multiple

width_multiple: 1.25 # layer channel multiple

anchors:

  - [10, 13, 16, 30, 33, 23] # P3/8

  - [30, 61, 62, 45, 59, 119] # P4/16

  - [116, 90, 156, 198, 373, 326] # P5/32



# YOLOv5 v6.0 backbone

backbone:

  # [from, number, module, args]

  [

    [-1, 1, Conv, [64, 6, 2, 2]], # 0-P1/2

    [-1, 1, Conv, [128, 3, 2]], # 1-P2/4

    [-1, 3, C3, [128]],

    [-1, 1, Conv, [256, 3, 2]], # 3-P3/8

    [-1, 6, C3, [256]],

    [-1, 1, Conv, [512, 3, 2]], # 5-P4/16

    [-1, 9, C3, [512]],

    [-1, 1, Conv, [1024, 3, 2]], # 7-P5/32

    [-1, 3, C3, [1024]],

    [-1, 1, SPPF, [1024, 5]], # 9

  ]



# YOLOv5 v6.0 head

head: [

    [-1, 1, Conv, [512, 1, 1]],

    [-1, 1, nn.Upsample, [None, 2, "nearest"]],

    [[-1, 6], 1, Concat, [1]], # cat backbone P4

    [-1, 3, C3, [512, False]], # 13



    [-1, 1, Conv, [256, 1, 1]],

    [-1, 1, nn.Upsample, [None, 2, "nearest"]],

    [[-1, 4], 1, Concat, [1]], # cat backbone P3

    [-1, 3, C3, [256, False]], # 17 (P3/8-small)



    [-1, 1, Conv, [256, 3, 2]],

    [[-1, 14], 1, Concat, [1]], # cat head P4

    [-1, 3, C3, [512, False]], # 20 (P4/16-medium)



    [-1, 1, Conv, [512, 3, 2]],

    [[-1, 10], 1, Concat, [1]], # cat head P5

    [-1, 3, C3, [1024, False]], # 23 (P5/32-large)



    [[17, 20, 23], 1, Detect, [nc, anchors]], # Detect(P3, P4, P5)

  ]

# Train Custom YOLOv5 Detector



### Next, we'll fire off training!





Here, we are able to pass a number of arguments:

- **img:** define input image size

- **batch:** determine batch size

- **epochs:** define the number of training epochs. (Note: often, 3000+ are common here!)

- **data:** set the path to our yaml file

- **cfg:** specify our model configuration

- **weights:** specify a custom path to weights. (Note: you can download weights from the Ultralytics Google Drive [folder](https://drive.google.com/open?id=1Drs_Aiu7xx6S-ix95f9kNsA6ueKRpN2J))

- **name:** result names

- **nosave:** only save the final checkpoint

- **cache:** cache images for faster training

In [ ]:
# train yolov5s on custom data for 200 epochs

# time its performance

# %%time

%cd /kaggle/working/yolov5/

!python train.py --img 416 --batch 16 --epochs 200 --data {dataset.location}/data.yaml --cfg ./models/custom_yolov5x.yaml --weights 'yolov5x.pt' --name yolov5s_results  --cache

# Evaluate Custom YOLOv5 Detector Performance

You can view the training graphs associated with a training job in the `/content/yolov5/runs/train/yolov5s_results/results.png` folder.



Training losses and performance metrics are also saved to Tensorboard and also to a logfile defined above with the **--name** flag when we train. In our case, we named this `yolov5s_results`.



Note from Glenn: Partially completed `results.txt` files can be plotted with `from utils.utils import plot_results; plot_results()`.

In [ ]:
from utils.plots import plot_results  # plot results.txt as results.png

Image(filename='/kaggle/working/yolov5/runs/train/yolov5s_results/results.png', width=1000)  # view results.png

### Curious? Visualize Our Training Data with Labels



After training starts, view `train*.jpg` images to see training images, labels and augmentation effects.



Note a mosaic dataloader is used for training (shown below), a new dataloading concept developed by Glenn Jocher and first featured in [YOLOv4](https://arxiv.org/abs/2004.10934).

In [ ]:
Image(filename='/kaggle/working/yolov5/runs/train/yolov5s_results/val_batch0_labels.jpg', width=900)

In [ ]:
# print out an augmented training example

print("GROUND TRUTH AUGMENTED TRAINING DATA:")

Image(filename='/kaggle/working/yolov5/runs/train/yolov5s_results/train_batch0.jpg', width=900)

# Run Inference With Trained Weights



Next, we can run inference with a pretrained checkpoint on all images in the `test/images` folder to understand how our model performs on our test set.

In [ ]:
# trained weights are saved by default in our weights folder

%ls runs/

In [ ]:
%ls runs/train/yolov5s_results/weights

In the snippet below, replace `Cash-Counter-10` with the name of the folder in which your dataset is stored.

In [ ]:
%cd /kaggle/working/yolov5/

!python detect.py --weights runs/train/yolov5s_results/weights/best.pt --img 416 --conf 0.4 --source /kaggle/working/yolov5/MineGuard-Thermal-15/test/images

In [ ]:
import glob

from IPython.display import Image, display



for imageName in glob.glob('/kaggle/working/yolov5/runs/detect/exp/*.jpg')[:10]: #assuming JPG

    display(Image(filename=imageName))